In [ ]:
!pip install ultralytics kagglehub opencv-python


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.7 MB/s eta 0:00:00


In [ ]:
import kagglehub

# Download dataset
path = kagglehub.dataset_download("harshwalia/birds-vs-drone-dataset")

print("Path to dataset files:", path)


100%|██████████| 78.1M/78.1M [00:00<00:00, 158MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/harshwalia/birds-vs-drone-dataset/versions/1


In [ ]:
import os

for root, dirs, files in os.walk(path):
    print(root)
    break


/root/.cache/kagglehub/datasets/harshwalia/birds-vs-drone-dataset/versions/1


In [ ]:
import os

dataset_root = os.path.join(path, "BirdVsDrone")
print(os.listdir(dataset_root))


['Drones', 'Birds']


In [ ]:
print(os.listdir(os.path.join(path, "BirdVsDrone")))


['Drones', 'Birds']


In [ ]:
import os

yolo_base = "/content/yolo_dataset"

folders = [
    "train/images", "train/labels",
    "val/images", "val/labels",
    "test/images", "test/labels"
]

for f in folders:
    os.makedirs(os.path.join(yolo_base, f), exist_ok=True)

print("✅ YOLO folder structure created")


✅ YOLO folder structure created


In [ ]:
import os

dataset_root = os.path.join(path, "BirdVsDrone")
birds_dir = os.path.join(dataset_root, "Birds")
drones_dir = os.path.join(dataset_root, "Drones")

print("Bird images:", len(os.listdir(birds_dir)))
print("Drone images:", len(os.listdir(drones_dir)))


Bird images: 400
Drone images: 428


In [ ]:
import random
import shutil

def process_class(class_dir, class_id):
    images = [img for img in os.listdir(class_dir) if img.lower().endswith(('.jpg','.png','.jpeg'))]
    random.shuffle(images)

    train_end = int(0.7 * len(images))
    val_end = int(0.85 * len(images))

    splits = {
        "train": images[:train_end],
        "val": images[train_end:val_end],
        "test": images[val_end:]
    }

    for split, imgs in splits.items():
        for img in imgs:
            src_img = os.path.join(class_dir, img)
            dst_img = os.path.join(yolo_base, split, "images", img)
            shutil.copy(src_img, dst_img)

            label_file = img.rsplit(".", 1)[0] + ".txt"
            dst_label = os.path.join(yolo_base, split, "labels", label_file)

            # YOLO format: class x_center y_center width height
            with open(dst_label, "w") as f:
                f.write(f"{class_id} 0.5 0.5 1.0 1.0")

# 0 = bird, 1 = drone
process_class(birds_dir, 0)
process_class(drones_dir, 1)

print("✅ Dataset converted to YOLO detection format")


✅ Dataset converted to YOLO detection format


In [ ]:
import glob

print("Train images:", len(glob.glob("/content/yolo_dataset/train/images/*")))
print("Val images:", len(glob.glob("/content/yolo_dataset/val/images/*")))
print("Test images:", len(glob.glob("/content/yolo_dataset/test/images/*")))


Train images: 577
Val images: 124
Test images: 125


In [ ]:
data_yaml = """
path: /content/yolo_dataset
train: train/images
val: val/images
test: test/images

names:
  0: bird
  1: drone
"""

with open("data.yaml", "w") as f:
    f.write(data_yaml)

print("✅ data.yaml created")


✅ data.yaml created


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0
)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.6 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7be8a48997f0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

--------------------------------------------------------


----------------------------------------------------------------------------------------------------------------------------


In [ ]:
!ls runs/detect


train


In [ ]:
for folder in ["train", "train2", "train3", "train4", "train5"]:
    print(folder, "->", end=" ")
    !ls runs/detect/{folder}/weights 2>/dev/null


train -> best.pt  last.pt
train2 -> train3 -> train4 -> train5 -> train7 -> 

In [ ]:
!ls runs/detect/train7/weights


ls: cannot access 'runs/detect/train7/weights': No such file or directory


In [ ]:
from ultralytics import YOLO
model = YOLO("runs/detect/train5/weights/best.pt")


FileNotFoundError: [Errno 2] No such file or directory: 'runs/detect/train5/weights/best.pt'

In [ ]:
metrics = model.val(data="data.yaml", imgsz=640)


In [ ]:
print("mAP@0.5      :", metrics.box.map50)
print("mAP@0.5:0.95 :", metrics.box.map)
print("Precision    :", metrics.box.mp)
print("Recall       :", metrics.box.mr)


In [ ]:
p = metrics.box.mp
r = metrics.box.mr
f1 = 2 * (p * r) / (p + r)

print("F1-score:", f1)


In [ ]:
import os
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

val_images = "/content/yolo_dataset/val/images"
val_labels = "/content/yolo_dataset/val/labels"

y_true = []
y_pred = []

for img in os.listdir(val_images):
    img_path = os.path.join(val_images, img)

    # remove extension safely (.jpg, .JPEG, .png, etc.)
    img_name = os.path.splitext(img)[0]
    label_path = os.path.join(val_labels, img_name + ".txt")

    # skip if label does not exist
    if not os.path.exists(label_path):
        continue

    # ground truth
    with open(label_path, "r") as f:
        gt_class = int(f.readline().split()[0])

    # prediction
    result = model(img_path, conf=0.25, verbose=False)[0]
    if len(result.boxes.cls) > 0:
        pred_class = int(result.boxes.cls[0])
    else:
        pred_class = 0  # default = bird

    y_true.append(gt_class)
    y_pred.append(pred_class)

# confusion matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=["Bird", "Drone"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()
